<a href="https://colab.research.google.com/github/Jinn-M4/Unified-Autonomous-Driving-System/blob/main/Unified_Autonomous_Driving_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

folders = [
    "simulation",
    "perception",
    "system",
    "planning",
    "control"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    open(f"{folder}/__init__.py", "w").close()  # 패키지 인식용

## Simulation

In [2]:
%%writefile simulation/vehicle.py
class Vehicle:
    def __init__(self):
        self.speed = 20.0
        self.lead_distance = 30.0
        self.lane_offset = 0.2
        self.lead_speed = 15.0

    def update(self, accel, steering):
        self.speed += accel * 0.1
        relative_speed = self.speed - self.lead_speed
        self.lead_distance -= relative_speed * 0.1
        self.lane_offset += steering * 0.1

        self.speed = max(0, self.speed)

        # 충돌 방지
        if self.lead_distance < 0:
           self.lead_distance = 0
           self.speed = 0


Writing simulation/vehicle.py


## Perception

In [3]:
%%writefile perception/sensors.py
import random

def get_lane_offset(vehicle):
    noise = random.uniform(-0.05, 0.05)
    return vehicle.lane_offset + noise

def get_lead_distance(vehicle):
    cam_noise = random.uniform(-2.0, 2.0)
    radar_noise = random.uniform(-1.0, 1.0)

    cam_dist = vehicle.lead_distance + cam_noise
    radar_dist = vehicle.lead_distance + radar_noise

    return cam_dist, radar_dist

Writing perception/sensors.py


## System

In [4]:
%%writefile system/fusion.py
def fuse_distance(cam_dist, radar_dist):
    return 0.7 * radar_dist + 0.3 * cam_dist

Writing system/fusion.py


## Planning

In [5]:
%%writefile planning/behavior.py

prev_state = "CRUISE"
state_duration = 0
MIN_STATE_DURATION = 3

def compute_ttc(distance, speed):
    if speed <= 0:
        return float('inf')
    return distance / speed


def decide_behavior(distance, speed):
    global prev_state, state_duration

    ttc = compute_ttc(distance, speed)

    # 기본 상태 결정
    if ttc < 1.5:
        new_state = "EMERGENCY_BRAKE"
    elif ttc < 3:
        new_state = "SLOW_DOWN"
    elif distance < 30:
        new_state = "FOLLOW"
    else:
        new_state = "CRUISE"

    # ✅ smoothing 적용
    if new_state != prev_state:
        if state_duration < MIN_STATE_DURATION:
            new_state = prev_state
        else:
            prev_state = new_state
            state_duration = 0

    state_duration += 1

    return prev_state

Writing planning/behavior.py


## Control

In [6]:
%%writefile control/acc.py
ACC_MAX = 2.5
BRAKE_NORMAL = -4.0
BRAKE_EMERGENCY = -9.0
TIME_HEADWAY = 2.2

# PID 파라미터
Kp = 1.5
Ki = 0.05
Kd = 0.2

integral_error = 0.0
prev_error = 0.0


def acc_control(distance, speed, behavior, dt=0.1):
    global integral_error, prev_error

    desired_distance = speed * TIME_HEADWAY
    error = distance - desired_distance

    if behavior == "EMERGENCY_BRAKE":
        acc = BRAKE_EMERGENCY

    elif behavior == "SLOW_DOWN":
        acc = BRAKE_NORMAL

    elif behavior == "FOLLOW":
        # PID 계산
        integral_error += error * dt

        # 🔥 windup 방지
        integral_error = max(-20, min(integral_error, 20))

        raw_derivative = (error - prev_error) / dt

        # 🔥 derivative smoothing
        alpha = 0.7
        prev_derivative = 0.0
        derivative = alpha * raw_derivative + (1 - alpha) * prev_derivative
        prev_derivative = derivative

        # 🔥 상태 기반 gain
        Kp_local = Kp * 1.3

        acc = Kp_local * error + Ki * integral_error + Kd * derivative

        prev_error = error

    else:  # CRUISE
        acc = ACC_MAX

    acc = max(BRAKE_EMERGENCY, min(acc, ACC_MAX))

    if speed < 0.1:
       return 0
    return acc

def acc_control_rule(distance, speed, behavior):
    desired_distance = speed * TIME_HEADWAY

    if behavior == "EMERGENCY_BRAKE":
        acc = BRAKE_EMERGENCY

    elif behavior == "SLOW_DOWN":
        acc = BRAKE_NORMAL

    elif behavior == "FOLLOW":
        acc = 0.1 * (distance - desired_distance)

    else:
        acc = ACC_MAX

    acc = max(BRAKE_EMERGENCY, min(acc, ACC_MAX))
    return acc


Writing control/acc.py


In [7]:
%%writefile control/lka.py
def lka_control(offset):
    k = 0.5
    return -k * offset

Writing control/lka.py


## Main

In [8]:
%%writefile main.py
import time
import csv
import matplotlib.pyplot as plt

from simulation.vehicle import Vehicle
from perception.sensors import get_lane_offset, get_lead_distance
from system.fusion import fuse_distance
from planning.behavior import decide_behavior
from control.acc import acc_control, acc_control_rule
from control.lka import lka_control
from control import acc as acc_module


def run_simulation(use_pid=True):
    vehicle = Vehicle()

    # PID 초기화 (중요)
    acc_module.integral_error = 0.0
    acc_module.prev_error = 0.0

    time_log = []
    speed_log = []
    distance_log = []
    behavior_log = []
    ttc_log = []

    for t in range(50):

        if t == 20:
            vehicle.lead_speed = 0

        lane_offset = get_lane_offset(vehicle)
        cam_dist, radar_dist = get_lead_distance(vehicle)

        fused_distance = fuse_distance(cam_dist, radar_dist)
        behavior = decide_behavior(fused_distance, vehicle.speed)

        ttc = fused_distance / vehicle.speed if vehicle.speed > 0 else float('inf')

        if use_pid:
            accel = acc_control(fused_distance, vehicle.speed, behavior)
        else:
            accel = acc_control_rule(fused_distance, vehicle.speed, behavior)

        steering = lka_control(lane_offset)
        vehicle.update(accel, steering)

        time_log.append(t)
        speed_log.append(vehicle.speed)
        distance_log.append(vehicle.lead_distance)
        behavior_log.append(behavior)
        ttc_log.append(ttc)

    return time_log, speed_log, distance_log, behavior_log, ttc_log


def analyze_performance(name, time_log, speed_log, distance_log, ttc_log):
    import numpy as np

    speed = np.array(speed_log)
    distance = np.array(distance_log)
    ttc = np.array(ttc_log)

    TIME_HEADWAY = 2.2
    desired_distance = speed * TIME_HEADWAY

    distance_error = np.abs(distance - desired_distance)
    mean_error = np.mean(distance_error)

    # ✅ inf 제거
    ttc = ttc[np.isfinite(ttc)]
    min_ttc = np.min(ttc) if len(ttc) > 0 else float('inf')

    accel = np.diff(speed)
    smoothness = np.mean(np.abs(accel))

    print(f"\n📊 [{name} Performance]")
    print(f"Mean Distance Error: {mean_error:.2f}")
    print(f"Minimum TTC: {min_ttc:.2f}")
    print(f"Speed Smoothness: {smoothness:.2f}")

    return {
        "mean_error": mean_error,
        "min_ttc": min_ttc,
        "smoothness": smoothness
    }


def save_csv(filename, time_log, speed_log, distance_log, behavior_log):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["time", "speed", "distance", "behavior"])
        for i in range(len(time_log)):
            writer.writerow([
                time_log[i],
                round(speed_log[i], 1),
                round(distance_log[i], 1),
                behavior_log[i]
            ])


def compare_results():
    pid_data = run_simulation(use_pid=True)
    rule_data = run_simulation(use_pid=False)

    t1, speed_pid, dist_pid, beh_pid, ttc_pid = pid_data
    t2, speed_rule, dist_rule, beh_rule, ttc_rule = rule_data

    pid_metrics = analyze_performance("PID", t1, speed_pid, dist_pid, ttc_pid)
    rule_metrics = analyze_performance("Rule-based", t2, speed_rule, dist_rule, ttc_rule)

    print("\n📈 Comparison Summary")

    def better(a, b, metric):
        if metric == "min_ttc":
            return "PID" if a > b else "Rule"
        else:
            return "PID" if a < b else "Rule"

    print(f"Better Distance Tracking: {better(pid_metrics['mean_error'], rule_metrics['mean_error'], 'error')}")
    print(f"Better Safety (TTC): {better(pid_metrics['min_ttc'], rule_metrics['min_ttc'], 'min_ttc')}")
    print(f"Better Smoothness: {better(pid_metrics['smoothness'], rule_metrics['smoothness'], 'smooth')}")

    save_csv("pid_log.csv", t1, speed_pid, dist_pid, beh_pid)
    save_csv("rule_log.csv", t2, speed_rule, dist_rule, beh_rule)

    print("✅ CSV 저장 완료")

    # ✅ 비교 그래프 (명확한 figure 분리)
    fig = plt.figure(figsize=(10, 5))

    plt.plot(t1, speed_pid, label="PID Speed")
    plt.plot(t2, speed_rule, linestyle='--', label="Rule Speed")

    plt.plot(t1, dist_pid, label="PID Distance", alpha=0.5)
    plt.plot(t2, dist_rule, linestyle='--', label="Rule Distance", alpha=0.5)

    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title("PID vs Rule-based ACC")
    plt.legend()

    plt.savefig("comparison_result.png")
    plt.show()
    plt.close(fig)   # 🔥 핵심


def plot_results(time_log, speed_log, distance_log, behavior_log, ttc_log):

    fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.plot(time_log, speed_log, label="Speed", linewidth=2)
    ax1.plot(time_log, distance_log, label="Distance", linestyle='--')
    ax1.set_xlabel("Time")
    ax1.set_ylabel("Speed / Distance")

    ax2 = ax1.twinx()
    ax2.plot(time_log, ttc_log, linestyle=':', label="TTC")
    ax2.set_ylabel("TTC (s)")

    color_map = {
        "CRUISE": "green",
        "FOLLOW": "blue",
        "SLOW_DOWN": "orange",
        "EMERGENCY_BRAKE": "red"
    }

    start = 0
    current_state = behavior_log[0]

    for i in range(1, len(time_log)):
        if behavior_log[i] != current_state:
            ax1.axvspan(start, i, color=color_map[current_state], alpha=0.1)
            start = i
            current_state = behavior_log[i]

    ax1.axvspan(start, len(time_log)-1, color=color_map[current_state], alpha=0.1)

    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")

    plt.title("Vehicle State with Behavior Segmentation")
    plt.savefig("simulation_result.png")

    plt.show()
    plt.close(fig)   # 🔥 핵심


if __name__ == "__main__":
    compare_results()
    data = run_simulation(use_pid=True)
    plot_results(*data)

Writing main.py


In [9]:
!python main.py


📊 [PID Performance]
Mean Distance Error: 4.52
Minimum TTC: 1.37
Speed Smoothness: 0.41

📊 [Rule-based Performance]
Mean Distance Error: 5.05
Minimum TTC: 1.50
Speed Smoothness: 0.34

📈 Comparison Summary
Better Distance Tracking: PID
Better Safety (TTC): Rule
Better Smoothness: Rule
✅ CSV 저장 완료
Figure(1000x500)
Figure(1000x500)
